# Overview-only plotting notebook: main baselines + APPLE/FedALA/PostALA

This notebook is intentionally minimal. It uses the clean merged manifest, but it only loads and plots the main old 5-client experiment methods:

- Fully Local
- FedAvg
- FedProx
- GCFL+
- FedALA-Full
- APPLE-Full
- APPLE-ALA
- APPLE-PostALA
- Local-Centralized Upper Bound

It does **not** generate expert/oracle/ablation comparison folders, method diagnostics, or extra head-to-head comparison tables.


In [ ]:

from __future__ import annotations

import math
import re
import json
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# =============================================================================
# User config
# =============================================================================

def find_project_root() -> Path:
    """Find the Subgraph_Federated_Learning project root from the current notebook path."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for cand in candidates:
        if (cand / "andrea" / "q_multihead_final_manifest.csv").exists():
            return cand
    # Fallback for Andrea's machine.
    fallback = Path("/Users/andreali/Documents/Subgraph_Federated_Learning")
    if (fallback / "andrea" / "q_multihead_final_manifest.csv").exists():
        return fallback
    raise FileNotFoundError(
        "Could not find andrea/q_multihead_final_manifest.csv. "
        "Run this notebook from the project root or set PROJECT_ROOT manually."
    )

PROJECT_ROOT = find_project_root()
MANIFEST_PATH = PROJECT_ROOT / "andrea" / "q_multihead_final_manifest.csv"
OUTPUT_ROOT = PROJECT_ROOT / "andrea" / "plots_final_multiselect_post_ala_overview_only"

OVERVIEW_DIR = OUTPUT_ROOT / "overview"
DIAGNOSTICS_DIR = OUTPUT_ROOT / "diagnostics"
TABLES_DIR = OUTPUT_ROOT / "tables_csv"
CACHE_DIR = OUTPUT_ROOT / "cache"

for d in [OVERVIEW_DIR, DIAGNOSTICS_DIR, TABLES_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# No loss selector.
SELECTION_METRICS = ["micro_f1", "macro_pos_f1", "micro_pr_auc", "macro_pr_auc"]

PROTOCOLS = {
    "OO": {
        "selection_protocol": "oracle_full",
        "label": "O+O: oracle/full validation selection → oracle/full test",
        "short_label": "O+O",
    },
    "VV": {
        "selection_protocol": "realistic_visible",
        "label": "V+V: visible validation selection → visible test",
        "short_label": "V+V",
    },
    "VO": {
        "selection_protocol": "realistic_selection_oracle",
        "label": "V+O: visible validation selection → oracle/full test",
        "short_label": "V+O",
    },
}

Q_LEVELS = {
    0.2: {"tag": "q02", "name": "iid", "title": "IID / low heterogeneity (q=0.2)"},
    0.5: {"tag": "q05", "name": "mid", "title": "Mid heterogeneity (q=0.5)"},
    0.8: {"tag": "q08", "name": "high", "title": "High heterogeneity (q=0.8)"},
}

TASKS = ["cycle2", "cycle3", "cycle4", "cycle5", "cycle6"]

METHOD_ORDER = [
    "fully_local_multi_select",
    "fedavg_multi_select",
    "fedprox_multi_select",
    "gcfl_plus_multi_select",
    "fedala_fedavg",
    "apple_backbone_taskhead",
    "apple_ala_multi_select",
    "apple_post_ala_multi_select",
    "local_centralized_multi_select",
]

METHOD_LABELS = {
    "fully_local_multi_select": "Fully Local",
    "local_centralized_multi_select": "Local-Centralized Upper Bound",
    "fedavg_multi_select": "FedAvg",
    "fedprox_multi_select": "FedProx",
    "gcfl_plus_multi_select": "GCFL+",

    # Canonical display names for full/vanilla-ish baselines.
    "fedala_fedavg": "FedALA-Full",
    "apple_backbone_taskhead": "APPLE-Full",

    # Main proposed / investigated methods.
    "apple_ala_multi_select": "APPLE-ALA",
    "apple_post_ala_multi_select": "APPLE-PostALA",

    # Ablations / adaptations.
    "fedala_head_only": "FedALA-Head",
    "apple_taskhead": "APPLE-Head",
    "apple_fedavg_backbone_taskhead": "FedAvgBackbone+APPLE-Head",

    # Oracle / diagnostic variants.
    "apple_fedavg_backbone_qinit_taskhead": "APPLE-QInit",
    "apple_fedavg_backbone_oracleq_taskhead": "APPLE-QOracle",

    # Expert variants kept after cleanup.
    "taskexpert_apple_multi_select": "TaskExpert-APPLE",
    "taskexpert_apple_expertprox_multi_select": "TaskExpert-APPLE-ExpertProx",
}

# Canonical method groups used for audience-facing focused plots.
# Main methods only: this is the exact audience-facing overview set.
MAIN_BASELINE_METHODS = [
    "fully_local_multi_select",
    "fedavg_multi_select",
    "fedprox_multi_select",
    "gcfl_plus_multi_select",
    "fedala_fedavg",
    "apple_backbone_taskhead",
    "apple_ala_multi_select",
    "apple_post_ala_multi_select",
    "local_centralized_multi_select",
]

FOCUSED_GROUPS = {
    "main_baselines": {
        "title": "Main baselines + APPLE/FedALA/PostALA",
        "methods": MAIN_BASELINE_METHODS,
        "dir_name": "overview_main_baselines_postala",
        "file_prefix": "main_baselines_postala",
    },
}

# Backward-compatible default focused order.
FOCUSED_METHOD_ORDER = MAIN_BASELINE_METHODS

# Baselines included in every method diagnostics PDF.
BASELINE_METHODS = ["fully_local_multi_select", "local_centralized_multi_select"]

# Heavy outputs. Switch any to False while debugging.
GENERATE_OVERVIEW_PDFS = False
GENERATE_METHOD_DIAGNOSTIC_PDFS = False
GENERATE_EXTRA_DR_ALA_DIAGNOSTICS = False

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MANIFEST_PATH:", MANIFEST_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


In [ ]:

# =============================================================================
# Path and manifest helpers
# =============================================================================

REMOTE_PROJECT_ROOTS = [
    "/home/nfs/ali7/Subgraph_Federated_Learning",
]


def is_blank(value) -> bool:
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except Exception:
        pass
    s = str(value).strip()
    return s == "" or s.lower() in {"nan", "none", "null"}


def resolve_path(value) -> Optional[Path]:
    """Resolve manifest path values to local paths under PROJECT_ROOT when needed."""
    if is_blank(value):
        return None
    s = str(value).strip()
    if s.startswith("./"):
        s = s[2:]
    for remote_root in REMOTE_PROJECT_ROOTS:
        prefix = remote_root.rstrip("/") + "/"
        if s.startswith(prefix):
            return PROJECT_ROOT / s[len(prefix):]
    p = Path(s)
    if p.is_absolute():
        return p
    return PROJECT_ROOT / p


def load_manifest() -> pd.DataFrame:
    manifest = pd.read_csv(MANIFEST_PATH)
    if "run_type" not in manifest.columns:
        raise ValueError("Manifest has no run_type column.")
    if "out_csv" not in manifest.columns and "local_out_csv_path" not in manifest.columns:
        raise ValueError("Manifest needs out_csv or local_out_csv_path.")

    if "local_out_csv_path" not in manifest.columns:
        manifest["local_out_csv_path"] = manifest["out_csv"].map(lambda x: str(resolve_path(x) or ""))
    else:
        manifest["local_out_csv_path"] = manifest["local_out_csv_path"].where(
            ~manifest["local_out_csv_path"].map(is_blank),
            manifest.get("out_csv", pd.Series([""] * len(manifest))).map(lambda x: str(resolve_path(x) or "")),
        )

    for col in ["dr_csv_path", "ala_csv_path", "clusters_csv_path"]:
        local_col = "local_" + col
        if col in manifest.columns and local_col not in manifest.columns:
            manifest[local_col] = manifest[col].map(lambda x: str(resolve_path(x) or ""))
        elif local_col in manifest.columns:
            manifest[local_col] = manifest[local_col].fillna("").astype(str)

    # Standardize numeric columns.
    if "q_value" in manifest.columns:
        manifest["q_value"] = pd.to_numeric(manifest["q_value"], errors="coerce").round(10)
    if "seed" in manifest.columns:
        manifest["seed"] = pd.to_numeric(manifest["seed"], errors="coerce").astype("Int64")

    # Keep only known/desired methods, in a stable order.
    manifest = manifest[manifest["run_type"].isin(METHOD_ORDER)].copy()
    manifest["method_label"] = manifest["run_type"].map(METHOD_LABELS).fillna(manifest["run_type"])

    manifest["local_out_csv_exists"] = manifest["local_out_csv_path"].map(lambda p: Path(p).exists() if str(p) else False)
    if not bool(manifest["local_out_csv_exists"].all()):
        missing = manifest.loc[~manifest["local_out_csv_exists"], ["run_type", "seed", "q_value", "out_csv", "local_out_csv_path"]]
        raise FileNotFoundError("Some out_csv files are missing locally:\n" + missing.to_string(index=False))

    return manifest.reset_index(drop=True)

manifest = load_manifest()
print("manifest rows:", len(manifest))
print("rows by run_type:")
print(manifest["run_type"].value_counts().reindex(METHOD_ORDER).dropna().astype(int).to_string())
print("\nrows by q_value:")
print(manifest["q_value"].value_counts().sort_index().to_string())
print("\nrows by actual run directory:")
print(manifest["local_out_csv_path"].map(lambda p: str(Path(p).parent)).value_counts().sort_index().to_string())


In [ ]:
# =============================================================================
# Manifest sanity checks for the overview-only method set
# =============================================================================

EXPECTED_METHOD_COUNTS = {
    "fully_local_multi_select": 45,
    "fedavg_multi_select": 9,
    "fedprox_multi_select": 9,
    "gcfl_plus_multi_select": 9,
    "fedala_fedavg": 9,
    "apple_backbone_taskhead": 9,
    "apple_ala_multi_select": 9,
    "apple_post_ala_multi_select": 9,
    "local_centralized_multi_select": 9,
}

EXPECTED_TOTAL_ROWS = 117
EXPECTED_Q_ROWS = 39

print("\nOverview-only manifest sanity check:")
method_counts = manifest["run_type"].value_counts().sort_index()

for method, expected in EXPECTED_METHOD_COUNTS.items():
    got = int(method_counts.get(method, 0))
    print(f"  {method}: {got} rows")
    if got != expected:
        raise ValueError(f"Expected {expected} rows for {method}, got {got}.")

unknown = sorted(set(method_counts.index) - set(EXPECTED_METHOD_COUNTS))
if unknown:
    raise ValueError("Unexpected methods after overview-only filtering: " + ", ".join(unknown))

if len(manifest) != EXPECTED_TOTAL_ROWS:
    raise ValueError(f"Expected {EXPECTED_TOTAL_ROWS} manifest rows, got {len(manifest)}.")
print(f"  total overview-only manifest rows: {EXPECTED_TOTAL_ROWS} OK")

q_counts = manifest["q_value"].value_counts().sort_index()
print("  q counts:")
print(q_counts.to_string())
if not all(int(q_counts.get(q, 0)) == EXPECTED_Q_ROWS for q in [0.2, 0.5, 0.8]):
    raise ValueError(f"Expected {EXPECTED_Q_ROWS} overview-only rows for each q-value.")
print("  q coverage: OK")


In [ ]:

# =============================================================================
# Load result CSVs efficiently
# =============================================================================

RESULT_WANTED_COLUMNS = [
    # identifiers
    "run_type", "algorithm", "subset_id", "subset_clients", "seed", "graph_id", "dataset_id",
    # row semantics
    "phase", "split", "task", "round", "local_epoch",
    "eval_mask_mode", "selection_protocol", "selected_by", "selector_metric",
    "selector_direction", "selected_by_eval_mode", "selected_round", "selected_epoch",
    "best_val_metric_value", "mean_selection_protocol",
    # metrics
    "train_loss", "eval_loss", "micro_f1", "macro_f1", "macro_pos_f1", "macro_minority_f1",
    "micro_pr_auc", "macro_pr_auc", "pr_auc", "positive_f1", "minority_f1",
    "tp", "fp", "tn", "fn", "precision", "recall", "f1", "pos_cnt", "pos_rate",
    "num_nodes", "visible_pairs", "total_pairs",
    # ALA diagnostics embedded in main result CSVs for APPLE-ALA variants
    "client_idx", "old_global_l2", "ala_steps", "ala_first_loss", "ala_last_loss",
    "ala_loss_std_window", "ala_skipped_identical", "ala_mean", "ala_std", "ala_min", "ala_max",
    "ala_head_mean", "ala_head_std", "ala_head_min", "ala_head_max",
    "ala_backbone_mean", "ala_backbone_std", "ala_backbone_min", "ala_backbone_max",
    "ala_task0_mean", "ala_task1_mean", "ala_task2_mean", "ala_task3_mean", "ala_task4_mean",
    "init_old_l2", "init_global_l2",
    # APPLE embedded diagnostics
    "base_train_loss", "dr_prox_loss", "apple_lambda", "apple_mu", "dr_row_sum",
    "dr_l2_to_p0", "dr_self_weight", "dr_backbone_row_sum", "dr_backbone_self_weight",
    "dr_backbone_l2_to_p0",
]


def read_csv_selected_columns(path: Path, wanted: Sequence[str]) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0)
    usecols = [c for c in wanted if c in header.columns]
    return pd.read_csv(path, usecols=usecols, low_memory=False)


def load_all_result_rows(manifest: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for idx, row in manifest.iterrows():
        p = Path(row["local_out_csv_path"])
        try:
            df = read_csv_selected_columns(p, RESULT_WANTED_COLUMNS)
        except Exception as e:
            raise RuntimeError(f"Could not read result CSV: {p}\n{type(e).__name__}: {e}") from e

        # Manifest values are the source of truth after merge normalization.
        df["_manifest_row"] = int(idx)
        df["_source_csv"] = str(p)
        df["run_type"] = row["run_type"]
        df["algorithm"] = row.get("algorithm", row["run_type"])
        df["method_label"] = METHOD_LABELS.get(row["run_type"], row["run_type"])
        df["q_value"] = float(row["q_value"]) if not pd.isna(row.get("q_value", np.nan)) else np.nan
        df["manifest_seed"] = int(row["seed"]) if not pd.isna(row.get("seed", np.nan)) else np.nan
        df["manifest_subset_clients"] = row.get("subset_clients", "")
        pieces.append(df)

        if (idx + 1) % 25 == 0 or (idx + 1) == len(manifest):
            print(f"loaded {idx + 1}/{len(manifest)} result CSVs")

    out = pd.concat(pieces, ignore_index=True, sort=False)

    # Clean dtypes.
    for col in ["round", "seed", "manifest_seed", "selected_round", "q_value"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    for col in ["task", "phase", "split", "eval_mask_mode", "selection_protocol", "selector_metric", "graph_id", "run_type"]:
        if col in out.columns:
            out[col] = out[col].astype("object")
    return out

all_rows = load_all_result_rows(manifest)
print("all_rows shape:", all_rows.shape)
print("methods loaded:", sorted(all_rows["run_type"].dropna().unique()))
print("selector metrics found:", sorted(all_rows.get("selector_metric", pd.Series(dtype=object)).dropna().astype(str).unique()))
print("selection protocols found:", sorted(all_rows.get("selection_protocol", pd.Series(dtype=object)).dropna().astype(str).unique()))


In [ ]:

# =============================================================================
# Aggregation utilities
# =============================================================================

def clean_task_series(s: pd.Series) -> pd.Series:
    return s.astype("object")


def is_task_null(s: pd.Series) -> pd.Series:
    return s.isna() | s.astype(str).str.strip().str.lower().isin(["", "nan", "none", "null"])


def close_q(series: pd.Series, q: float) -> pd.Series:
    return np.isclose(pd.to_numeric(series, errors="coerce"), float(q), atol=1e-8)


def f1_from_counts(tp: float, fp: float, fn: float) -> float:
    denom = 2.0 * tp + fp + fn
    return float(2.0 * tp / denom) if denom > 0 else float("nan")


def weighted_mean(values: pd.Series, weights: Optional[pd.Series] = None) -> float:
    v = pd.to_numeric(values, errors="coerce")
    mask = v.notna()
    if weights is not None:
        w = pd.to_numeric(weights, errors="coerce")
        mask = mask & w.notna() & (w > 0)
        if mask.any():
            return float(np.average(v[mask].astype(float), weights=w[mask].astype(float)))
    if mask.any():
        return float(v[mask].mean())
    return float("nan")


def count_based_f1_or_weighted(rows: pd.DataFrame, value_col: str, weight_col: Optional[str] = None) -> float:
    if rows.empty:
        return float("nan")
    required = {"tp", "fp", "fn"}
    if required.issubset(rows.columns):
        counts = rows[["tp", "fp", "fn"]].apply(pd.to_numeric, errors="coerce")
        if counts.notna().all(axis=None):
            return f1_from_counts(counts["tp"].sum(), counts["fp"].sum(), counts["fn"].sum())
    weights = rows[weight_col] if weight_col and weight_col in rows.columns else None
    return weighted_mean(rows[value_col], weights)


def weight_column_for_protocol(protocol_key: str) -> str:
    return "visible_pairs" if protocol_key == "VV" else "total_pairs"


def selected_rows_for_seed(
    df: pd.DataFrame,
    *,
    method: str,
    q: float,
    seed: int,
    protocol_key: str,
    selector_metric: str,
    split: str = "test",
) -> pd.DataFrame:
    protocol = PROTOCOLS[protocol_key]["selection_protocol"]
    sub = df[
        (df["run_type"] == method)
        & close_q(df["q_value"], q)
        & (pd.to_numeric(df["manifest_seed"], errors="coerce") == int(seed))
        & (df["split"].astype(str) == split)
    ].copy()
    if "selection_protocol" not in sub.columns or "selector_metric" not in sub.columns:
        return sub.iloc[0:0]
    sub = sub[
        (sub["selection_protocol"].astype(str) == protocol)
        & (sub["selector_metric"].astype(str) == selector_metric)
    ].copy()
    return sub


def compute_seed_metrics(
    selected: pd.DataFrame,
    *,
    protocol_key: str,
) -> Dict[str, float]:
    """Compute one pooled seed-level result from selected test rows."""
    out: Dict[str, float] = {}
    if selected.empty:
        for key in ["micro_f1", "macro_pos_f1", "micro_pr_auc", "macro_pr_auc", *[f"{t}_pos_f1" for t in TASKS]]:
            out[key] = float("nan")
        out["selected_round"] = float("nan")
        out["n_client_rows"] = 0
        return out

    scalar = selected[is_task_null(selected.get("task", pd.Series([np.nan] * len(selected), index=selected.index)))].copy()
    task_rows = selected[~is_task_null(selected.get("task", pd.Series([np.nan] * len(selected), index=selected.index)))].copy()

    # Exclude mean rows when they appear with graph_id='all' and no selection_protocol. The selected_protocol filter usually removes them.
    scalar = scalar[scalar["selection_protocol"].notna()] if "selection_protocol" in scalar.columns else scalar
    task_rows = task_rows[task_rows["selection_protocol"].notna()] if "selection_protocol" in task_rows.columns else task_rows

    wcol = weight_column_for_protocol(protocol_key)
    if wcol not in scalar.columns:
        wcol = "num_nodes" if "num_nodes" in scalar.columns else None

    out["micro_f1"] = count_based_f1_or_weighted(scalar, "micro_f1", wcol)
    out["micro_pr_auc"] = weighted_mean(scalar.get("micro_pr_auc", pd.Series(dtype=float)), scalar[wcol] if wcol else None)
    out["macro_pr_auc"] = weighted_mean(scalar.get("macro_pr_auc", pd.Series(dtype=float)), scalar[wcol] if wcol else None)

    task_f1s = []
    for task in TASKS:
        tdf = task_rows[task_rows["task"].astype(str) == task]
        if not tdf.empty:
            val = count_based_f1_or_weighted(tdf, "positive_f1", "pos_cnt" if "pos_cnt" in tdf.columns else None)
        else:
            val = float("nan")
        out[f"{task}_pos_f1"] = val
        if not math.isnan(val):
            task_f1s.append(val)
    out["macro_pos_f1"] = float(np.mean(task_f1s)) if task_f1s else weighted_mean(scalar.get("macro_pos_f1", pd.Series(dtype=float)), scalar[wcol] if wcol else None)

    if "selected_round" in scalar.columns and scalar["selected_round"].notna().any():
        out["selected_round"] = float(pd.to_numeric(scalar["selected_round"], errors="coerce").mean())
    else:
        out["selected_round"] = float("nan")
    out["n_client_rows"] = int(len(scalar))
    return out


def seed_metric_rows(method: str, q: float, protocol_key: str, selector_metric: str) -> pd.DataFrame:
    seeds = sorted(pd.to_numeric(manifest.loc[(manifest["run_type"] == method) & close_q(manifest["q_value"], q), "seed"], errors="coerce").dropna().astype(int).unique())
    records = []
    for seed in seeds:
        selected = selected_rows_for_seed(
            all_rows,
            method=method,
            q=q,
            seed=seed,
            protocol_key=protocol_key,
            selector_metric=selector_metric,
        )
        metrics = compute_seed_metrics(selected, protocol_key=protocol_key)
        metrics.update({"run_type": method, "method": METHOD_LABELS.get(method, method), "q_value": q, "protocol": protocol_key, "selector_metric": selector_metric, "seed": seed})
        records.append(metrics)
    return pd.DataFrame(records)


def summarize_method_for_table(method: str, q: float, protocol_key: str, selector_metric: str) -> Dict:
    seed_df = seed_metric_rows(method, q, protocol_key, selector_metric)
    out = {"run_type": method, "Method": METHOD_LABELS.get(method, method), "q_value": q, "protocol": protocol_key, "selector_metric": selector_metric}
    metric_cols = ["micro_f1", "macro_pos_f1", "micro_pr_auc", "macro_pr_auc"] + [f"{t}_pos_f1" for t in TASKS] + ["selected_round"]
    for col in metric_cols:
        vals = pd.to_numeric(seed_df.get(col, pd.Series(dtype=float)), errors="coerce").dropna()
        out[f"{col}_mean"] = float(vals.mean()) if len(vals) else float("nan")
        out[f"{col}_std"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0 if len(vals) == 1 else float("nan")
    out["n_seeds"] = int(seed_df["seed"].nunique()) if "seed" in seed_df.columns else 0
    out["n_client_rows_mean"] = float(pd.to_numeric(seed_df.get("n_client_rows", pd.Series(dtype=float)), errors="coerce").mean()) if len(seed_df) else float("nan")
    return out


def fmt_pct(mean: float, std: float) -> str:
    if pd.isna(mean):
        return "—"
    if pd.isna(std):
        std = 0.0
    return f"{100*mean:.1f}% ± {100*std:.1f}%"


def fmt_num(mean: float, std: float, digits: int = 1) -> str:
    if pd.isna(mean):
        return "—"
    if pd.isna(std):
        std = 0.0
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


def build_table(q: float, protocol_key: str, selector_metric: str, methods: Optional[Sequence[str]] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    methods = list(methods or [m for m in METHOD_ORDER if m in set(manifest["run_type"])])
    raw_records = [summarize_method_for_table(method, q, protocol_key, selector_metric) for method in methods]
    raw = pd.DataFrame(raw_records)

    formatted = pd.DataFrame({"Method": raw["Method"]})
    formatted["Micro-F1"] = [fmt_pct(m, s) for m, s in zip(raw["micro_f1_mean"], raw["micro_f1_std"])]
    formatted["Macro Pos-F1"] = [fmt_pct(m, s) for m, s in zip(raw["macro_pos_f1_mean"], raw["macro_pos_f1_std"])]
    formatted["Micro PR-AUC"] = [fmt_pct(m, s) for m, s in zip(raw["micro_pr_auc_mean"], raw["micro_pr_auc_std"])]
    formatted["Macro PR-AUC"] = [fmt_pct(m, s) for m, s in zip(raw["macro_pr_auc_mean"], raw["macro_pr_auc_std"])]
    for task in TASKS:
        formatted[f"{task} pos-F1"] = [fmt_pct(m, s) for m, s in zip(raw[f"{task}_pos_f1_mean"], raw[f"{task}_pos_f1_std"])]
    formatted["Sel. round"] = [fmt_num(m, s, digits=1) for m, s in zip(raw["selected_round_mean"], raw["selected_round_std"])]
    formatted["n"] = raw["n_seeds"].astype(int).astype(str)
    return raw, formatted

# Quick smoke check for one table.
raw_demo, fmt_demo = build_table(0.8, "VO", "macro_pr_auc")
display(fmt_demo.head())
print("demo raw rows:", len(raw_demo))


In [ ]:

# =============================================================================
# Save all 36 tables as CSV
# =============================================================================

all_tables_raw: Dict[Tuple[float, str, str], pd.DataFrame] = {}
all_tables_fmt: Dict[Tuple[float, str, str], pd.DataFrame] = {}

for q in Q_LEVELS:
    for selector in SELECTION_METRICS:
        for proto_key in PROTOCOLS:
            raw, fmt = build_table(q, proto_key, selector)
            key = (q, selector, proto_key)
            all_tables_raw[key] = raw
            all_tables_fmt[key] = fmt

            qtag = Q_LEVELS[q]["tag"]
            raw_path = TABLES_DIR / f"{qtag}_{selector}_{proto_key}_raw.csv"
            fmt_path = TABLES_DIR / f"{qtag}_{selector}_{proto_key}_formatted.csv"
            raw.to_csv(raw_path, index=False)
            fmt.to_csv(fmt_path, index=False)

print(f"Saved {len(all_tables_raw)} raw tables and {len(all_tables_fmt)} formatted tables to:")
print(TABLES_DIR)


In [ ]:

# =============================================================================
# Plotting helpers
# =============================================================================

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "font.size": 8,
})


def safe_filename(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")


def add_title(fig, title: str, subtitle: Optional[str] = None):
    if subtitle:
        fig.suptitle(title + "\n" + subtitle, fontsize=11, fontweight="bold", y=0.985)
    else:
        fig.suptitle(title, fontsize=11, fontweight="bold", y=0.985)


def render_table_page(pdf: PdfPages, formatted: pd.DataFrame, title: str, subtitle: str = ""):
    fig, ax = plt.subplots(figsize=(17, 10))
    ax.axis("off")
    add_title(fig, title, subtitle)

    table = ax.table(
        cellText=formatted.values,
        colLabels=formatted.columns,
        loc="center",
        cellLoc="center",
        colLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(6.3)
    table.scale(1, 1.35)

    # Emphasize header and method column.
    for (r, c), cell in table.get_celld().items():
        if r == 0:
            cell.set_text_props(weight="bold")
        if c == 0 and r > 0:
            cell.set_text_props(weight="bold")

    fig.tight_layout(rect=[0, 0, 1, 0.93])
    pdf.savefig(fig)
    plt.close(fig)


def render_bar_page(pdf: PdfPages, raw: pd.DataFrame, title: str, subtitle: str = ""):
    metrics = [
        ("micro_f1", "Micro-F1"),
        ("macro_pos_f1", "Macro Pos-F1"),
        ("micro_pr_auc", "Micro PR-AUC"),
        ("macro_pr_auc", "Macro PR-AUC"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(17, 10))
    axes = axes.ravel()
    add_title(fig, title, subtitle)

    xlabels = raw["Method"].tolist()
    x = np.arange(len(xlabels))
    for ax, (metric, label) in zip(axes, metrics):
        mean = pd.to_numeric(raw[f"{metric}_mean"], errors="coerce") * 100.0
        std = pd.to_numeric(raw[f"{metric}_std"], errors="coerce") * 100.0
        ax.bar(x, mean, yerr=std, capsize=2)
        ax.set_title(label)
        ax.set_ylim(0, max(100, np.nanmax(mean + std) + 5 if np.isfinite(mean + std).any() else 100))
        ax.set_ylabel("%")
        ax.set_xticks(x)
        ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=7)
        ax.grid(axis="y", alpha=0.25)

    fig.tight_layout(rect=[0, 0, 1, 0.92])
    pdf.savefig(fig)
    plt.close(fig)


def plot_line_with_band(ax, summary: pd.DataFrame, x_col: str, y_mean: str, y_std: str, label: str):
    if summary.empty or y_mean not in summary.columns:
        return
    x = pd.to_numeric(summary[x_col], errors="coerce")
    y = pd.to_numeric(summary[y_mean], errors="coerce")
    s = pd.to_numeric(summary[y_std], errors="coerce").fillna(0.0)
    mask = x.notna() & y.notna()
    if not mask.any():
        return
    x = x[mask].astype(float).to_numpy()
    y = y[mask].astype(float).to_numpy()
    s = s[mask].astype(float).to_numpy()
    ax.plot(x, y, label=label)
    ax.fill_between(x, y - s, y + s, alpha=0.15)


In [ ]:

# =============================================================================
# Overview PDFs: one PDF per q-level, all selectors/protocols
# =============================================================================

def generate_overview_pdfs():
    created = []
    for q, qinfo in Q_LEVELS.items():
        pdf_path = OVERVIEW_DIR / f"overview_{qinfo['name']}_{qinfo['tag']}_all_selection_metrics.pdf"
        with PdfPages(pdf_path) as pdf:
            for selector in SELECTION_METRICS:
                for proto_key, proto in PROTOCOLS.items():
                    key = (q, selector, proto_key)
                    raw = all_tables_raw[key]
                    fmt = all_tables_fmt[key]
                    title = f"{qinfo['title']} | selector={selector} | protocol={proto_key}"
                    subtitle = proto["label"]
                    render_table_page(pdf, fmt, title, subtitle)
                    render_bar_page(pdf, raw, title + " | overview bars", subtitle)
        created.append(pdf_path)
        print("saved overview PDF ->", pdf_path)
    return created

if GENERATE_OVERVIEW_PDFS:
    overview_pdfs = generate_overview_pdfs()
else:
    overview_pdfs = []


In [ ]:
# =============================================================================
# Audience-facing overview PDFs: main baselines + APPLE/FedALA/PostALA only
# =============================================================================

GENERATE_FOCUSED_OVERVIEW_PDFS = True

focused_tables_raw: Dict[Tuple[str, float, str, str], pd.DataFrame] = {}
focused_tables_fmt: Dict[Tuple[str, float, str, str], pd.DataFrame] = {}

for group_key, group in FOCUSED_GROUPS.items():
    for q in Q_LEVELS:
        for selector in SELECTION_METRICS:
            for proto_key in PROTOCOLS:
                methods = [m for m in group["methods"] if m in set(manifest["run_type"])]
                raw, fmt = build_table(q, proto_key, selector, methods=methods)
                focused_tables_raw[(group_key, q, selector, proto_key)] = raw
                focused_tables_fmt[(group_key, q, selector, proto_key)] = fmt


def generate_focused_overview_pdfs():
    created = []
    for group_key, group in FOCUSED_GROUPS.items():
        group_dir = OUTPUT_ROOT / group["dir_name"]
        group_dir.mkdir(parents=True, exist_ok=True)
        for q, qinfo in Q_LEVELS.items():
            pdf_path = group_dir / f"{group['file_prefix']}_{qinfo['name']}_{qinfo['tag']}_all_selection_metrics.pdf"
            with PdfPages(pdf_path) as pdf:
                for selector in SELECTION_METRICS:
                    for proto_key, proto in PROTOCOLS.items():
                        key = (group_key, q, selector, proto_key)
                        raw = focused_tables_raw[key]
                        fmt = focused_tables_fmt[key]
                        title = f"{group['title']} | {qinfo['title']} | selector={selector} | protocol={proto_key}"
                        subtitle = proto["label"]
                        render_table_page(pdf, fmt, title, subtitle)
                        render_bar_page(pdf, raw, title + " | overview bars", subtitle)
            created.append(pdf_path)
            print("saved focused overview PDF ->", pdf_path)
    return created

if GENERATE_FOCUSED_OVERVIEW_PDFS:
    focused_overview_pdfs = generate_focused_overview_pdfs()
else:
    focused_overview_pdfs = []


In [ ]:
# =============================================================================
# Final output summary
# =============================================================================

print("\nDone.")
print("Output root:", OUTPUT_ROOT)

print("\nMain overview PDFs:")
for p in globals().get("focused_overview_pdfs", []):
    print(" -", p)

print("\nTables CSV folder:", TABLES_DIR)
print("Number of formatted table CSVs:", len(list(TABLES_DIR.glob("*_formatted.csv"))))
print("Number of raw table CSVs:", len(list(TABLES_DIR.glob("*_raw.csv"))))

print("\nThis notebook intentionally does not generate:")
print(" - expert/oracle/ablation focused overview folders")
print(" - method diagnostics PDFs")
print(" - head-to-head win-summary comparison CSVs")
